# Histopathology + ELM — Colab runner

Runtime → Change runtime type → **GPU**. Then run the cells top to bottom.

In [ ]:
!git clone -q https://github.com/ozlemkocc/histopathology-elm.git
%cd histopathology-elm
!pip install -q -e .

## 1. Download a dataset

- **LC25000, BreakHis:** Kaggle via `kagglehub`.
- **BACH:** Hugging Face (`1aurent/BACH`, the source used in the project). Images are exported into class folders; samples labelled *Unknown* are skipped.

In [ ]:
DATASET = "lc25000"  # "lc25000" | "breakhis" | "bach"

if DATASET == "bach":
    !pip install -q datasets
    from pathlib import Path
    from datasets import ClassLabel, Image as HFImage, load_dataset

    DATA_DIR = "/content/bach"
    ds = load_dataset("1aurent/BACH", split="train")
    image_col = next(k for k, v in ds.features.items() if isinstance(v, HFImage))
    label_col = next(k for k, v in ds.features.items() if isinstance(v, ClassLabel))
    names = ds.features[label_col].names
    folder = {"benign": "Benign", "insitu": "InSitu", "invasive": "Invasive", "normal": "Normal"}
    for i, row in enumerate(ds):
        key = names[row[label_col]].lower().replace(" ", "").replace("_", "")
        if key not in folder:  # e.g. "unknown"
            continue
        out = Path(DATA_DIR) / "Photos" / folder[key] / f"{i:04d}.png"
        out.parent.mkdir(parents=True, exist_ok=True)
        row[image_col].convert("RGB").save(out)
    print({c: len(list((Path(DATA_DIR) / 'Photos' / c).glob('*.png'))) for c in folder.values()})
else:
    import kagglehub
    KAGGLE = {
        "lc25000": "andrewmvd/lung-and-colon-cancer-histopathological-images",
        "breakhis": "ambarish/breakhis",
    }
    DATA_DIR = kagglehub.dataset_download(KAGGLE[DATASET])
print(DATA_DIR)

## 2. Train and evaluate all models

Deep features are cached in `results/<dataset>/features/`, so re-running with other PCA / ELM settings only takes seconds.

In [ ]:
!python -m histo_elm.run --dataset {DATASET} --data-dir "{DATA_DIR}" --out-dir results/{DATASET} --save-models

## 3. Results

In [ ]:
import pandas as pd
from IPython.display import Image, display

display(pd.read_csv(f"results/{DATASET}/summary.csv").round(4))
display(Image(f"results/{DATASET}/summary.png"))
display(Image(f"results/{DATASET}/confusion_matrices/deit-b.png"))

## 4. Predict a single image

In [ ]:
import glob
sample = sorted(glob.glob(f"{DATA_DIR}/**/*.*", recursive=True))[0]
!python -m histo_elm.predict results/{DATASET}/models/deit-b.joblib "{sample}"